In [23]:
%pip install requests gTTS deep-translator SpeechRecognition pyaudio

Note: you may need to restart the kernel to use updated packages.


In [24]:
import requests
import speech_recognition as sr
from gtts import gTTS
from deep_translator import MyMemoryTranslator
from IPython.display import Audio, display
from deep_translator import GoogleTranslator

In [25]:
def VoiceToText():
    recognizer = sr.Recognizer()

    with sr.Microphone() as source:
        print("Listening... Speak the city name!")
        recognizer.adjust_for_ambient_noise(source, duration=1)
        audio = recognizer.listen(source)

    try:
        text = recognizer.recognize_google(audio)
        print("You said:", text)
        return text.strip()

    except sr.UnknownValueError:
        print("Sorry, I could not understand.")
        return ""

    except sr.RequestError:
        print("Speech recognition service unavailable.")
        return ""

In [59]:
API_KEY = "your_api_key"

def get_weather(city):

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()

        if response.status_code != 200:
            print("Weather error:", data.get("message", "Unknown error"))
            return None

        city_name = data["name"]
        temperature = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        description = data["weather"][0]["description"]

        statement = (
            f"The current weather in {city_name} is {description}. "
            f"The temperature is {temperature} degrees Celsius. "
            f"The humidity is {humidity} percent."
        )

        return statement

    except requests.RequestException as e:
        print("Connection error:", e)
        return None

In [27]:
def translate_text(text, lang):
    if lang == "en":
        return text

    translated_text = GoogleTranslator(source="en", target="te").translate(text)

    if not translated_text:
        raise ValueError("Telugu translation was empty.")

    return translated_text

In [28]:
def text_to_voice(text, language="te"):

    tts = gTTS(text=text, lang=language, slow=False)

    tts.save("weather.mp3")

    display(Audio("weather.mp3", autoplay=True))

In [58]:
print("🌦️ LOCATION-BASED WEATHER ANNOUNCEMENT SYSTEM")
print("1. Voice Input")
print("2. Text Input")

choice = input("Choose input method (1 or 2): ")

if choice == "1":
    city = VoiceToText()

elif choice == "2":
    city = input("Enter city name: ").strip()

else:
    print("Invalid choice!")
    city = ""

if city:

    weather_statement = get_weather(city)

    if weather_statement:
        print("\nWeather Information:")
        print(weather_statement)

        
        text_to_voice(weather_statement, language="en")

🌦️ LOCATION-BASED WEATHER ANNOUNCEMENT SYSTEM
1. Voice Input
2. Text Input


Choose input method (1 or 2):  1


Listening... Speak the city name!
You said: Tanuku

Weather Information:
The current weather in Tanuku is clear sky. The temperature is 26.03 degrees Celsius. The humidity is 85 percent.
